# Paper-linked differential pair-rule results

## Setup

In [ ]:
from pathlib import Path
import sys
import warnings

ROOT = next(
    folder for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (folder / "result_summary" / "data_helper.py").exists()
)
sys.path[:0] = [
    str(ROOT / "result_summary"),
    str(ROOT / "result_summary" / "differential_rules"),
]

import data_helper as dh
import differential_stats as ds
import differential_vis as dv

warnings.filterwarnings("ignore")

STAGES = ["Control", "Mild", "Severe"]
ORGANS = ["Colon", "Duodenum"]
SCORES = ["Pathological score", "Clinical score"]
METRIC = "Lift"
METRIC_REFERENCE = 1
MIN_CELLS = 20
MIN_ELIGIBLE = 3
MIN_PRESENT = 3
N_PERMUTATIONS = 5_000

## Data

In [ ]:
df_cells, df_fovs, df_biopsy = dh.load_spatial_data()
df_results = dh.load_results(rule_max_items=2, kind=None)
rules = dh.prepare_rules(df_results)
rule_states, eligibility, eligible_states = ds.state_tables(
    rules, df_cells, df_fovs["FOV"], MIN_CELLS
)
print(f"{len(rules)} rows; {rules['Clean_Rule'].nunique()} directional pair rules")

In [ ]:
screens = {
    (organ, score): ds.severity_screen(
        eligible_states, df_fovs, organ, score, STAGES,
        min_eligible=MIN_ELIGIBLE,
        min_present=MIN_PRESENT,
        n_permutations=N_PERMUTATIONS,
    )
    for organ in ORGANS
    for score in SCORES
}
trend_results = {scope: result[0] for scope, result in screens.items()}
pair_results = {scope: result[1] for scope, result in screens.items()}

## Results

### 1. Duodenum epithelial rules — pathological score

In [ ]:
EPITHELIAL_SPECS = [
    {"rule": "Epithelial -> Goblet", "organ": "Duodenum", "score": "Pathological score"},
    {"rule": "Goblet -> Epithelial", "organ": "Duodenum", "score": "Pathological score"},
    {"rule": "Paneth -> Epithelial", "organ": "Duodenum", "score": "Pathological score"},
]
for spec in EPITHELIAL_SPECS:
    dv.plot_rule_states(
        rule_states, eligibility, df_fovs, [spec], STAGES,
        heading=f"Duodenum {spec['rule'].replace(' -> ', ' → ')} rule states",
        trend_results=trend_results, pair_results=pair_results,
    )
    dv.plot_rule_metric(
        rules, eligibility, df_fovs, spec, STAGES,
        metric=METRIC, reference=METRIC_REFERENCE,
    )


### 2. Colon plasma rules — clinical score

In [ ]:
COLON_PLASMA_SPECS = [
    {"rule": "CD4T -> Plasma", "organ": "Colon", "score": "Clinical score"},
    {"rule": "CD8T -> Plasma", "organ": "Colon", "score": "Clinical score"},
]
COLON_PLASMA_EXPORTS = {
    "CD4T -> Plasma": "tex_paper_colon_cd4_plasma.pdf",
    "CD8T -> Plasma": "tex_paper_colon_cd8_plasma.pdf",
}
for spec in COLON_PLASMA_SPECS:
    dv.plot_rule_and_cell_changes(
        rule_states, eligibility, df_cells, df_fovs, [spec], STAGES,
        heading=f"Colon {spec['rule'].replace(' -> ', ' → ')} beside cell abundance",
        trend_results=trend_results, pair_results=pair_results,
        save=COLON_PLASMA_EXPORTS[spec["rule"]],
    )
    dv.plot_rule_metric(
        rules, eligibility, df_fovs, spec, STAGES,
        metric=METRIC, reference=METRIC_REFERENCE,
    )


### 3. Duodenum plasma rules — clinical score

In [ ]:
DUODENUM_PLASMA_SPECS = [
    {"rule": "CD4T -> Plasma", "organ": "Duodenum", "score": "Clinical score"},
    {"rule": "Macrophage -> Plasma", "organ": "Duodenum", "score": "Clinical score"},
]
for spec in DUODENUM_PLASMA_SPECS:
    dv.plot_rule_states(
        rule_states, eligibility, df_fovs, [spec], STAGES,
        heading=f"Duodenum {spec['rule'].replace(' -> ', ' → ')} rule states",
        trend_results=trend_results, pair_results=pair_results,
    )
    dv.plot_rule_metric(
        rules, eligibility, df_fovs, spec, STAGES,
        metric=METRIC, reference=METRIC_REFERENCE,
    )


### 4. Colon immune and stromal rules — pathological score

In [ ]:
COLON_IMMUNE_STROMAL_SPECS = [
    {"rule": "Macrophage -> CD4T", "organ": "Colon", "score": "Pathological score"},
    {"rule": "Fibroblast -> Plasma", "organ": "Colon", "score": "Pathological score"},
]
for spec in COLON_IMMUNE_STROMAL_SPECS:
    dv.plot_rule_states(
        rule_states, eligibility, df_fovs, [spec], STAGES,
        heading=f"Colon {spec['rule'].replace(' -> ', ' → ')} rule states",
        trend_results=trend_results, pair_results=pair_results,
    )
    dv.plot_rule_metric(
        rules, eligibility, df_fovs, spec, STAGES,
        metric=METRIC, reference=METRIC_REFERENCE,
    )


### 5. Duodenum Paneth rules — pathological score

In [ ]:
PANETH_RULES = [
    ({"rule": "Paneth -> Epithelial", "organ": "Duodenum", "score": "Pathological score"},
     "tex_paper_paneth_to_epithelial.pdf"),
    ({"rule": "Epithelial -> Paneth", "organ": "Duodenum", "score": "Pathological score"},
     "tex_paper_epithelial_to_paneth.pdf"),
]
for spec, filename in PANETH_RULES:
    dv.plot_rule_with_cell_count(
        rule_states, eligibility, df_cells, df_fovs, spec, "Paneth", STAGES,
        heading=f"Paneth loss and {spec['rule'].replace(' -> ', ' → ')}",
        trend_results=trend_results, pair_results=pair_results, save=filename,
    )
    dv.plot_rule_metric(
        rules, eligibility, df_fovs, spec, STAGES,
        metric=METRIC, reference=METRIC_REFERENCE,
    )


### 6. Duodenum Endocrine rule — pathological score

In [ ]:
ENDOCRINE_EPITHELIAL = {"rule": "Endocrine -> Epithelial",
                         "organ": "Duodenum",
                         "score": "Pathological score"}
dv.plot_rule_with_cell_count(
    rule_states, eligibility, df_cells, df_fovs, ENDOCRINE_EPITHELIAL, "Endocrine",
    STAGES, heading="Endocrine accumulation and its epithelial rule",
    trend_results=trend_results, pair_results=pair_results,
    save="tex_paper_endocrine_context.pdf",
)
dv.plot_rule_metric(
    rules, eligibility, df_fovs, ENDOCRINE_EPITHELIAL, STAGES,
    metric=METRIC, reference=METRIC_REFERENCE,
)


### 7. Duodenum CD8T–epithelial rules

In [ ]:
CD8_EPITHELIAL_SPECS = [
    {"rule": "CD8T -> Goblet", "organ": "Duodenum", "score": "Pathological score"},
    {"rule": "Goblet -> CD8T", "organ": "Duodenum", "score": "Pathological score"},
    {"rule": "CD8T -> Epithelial", "organ": "Duodenum", "score": "Clinical score"},
]
for spec in CD8_EPITHELIAL_SPECS:
    dv.plot_rule_states(
        rule_states, eligibility, df_fovs, [spec], STAGES,
        heading=f"Duodenum {spec['rule'].replace(' -> ', ' → ')} rule states",
        trend_results=trend_results, pair_results=pair_results,
    )
    dv.plot_rule_metric(
        rules, eligibility, df_fovs, spec, STAGES,
        metric=METRIC, reference=METRIC_REFERENCE,
    )
